# AI-IO on T-Lab (Colab, resumable)
Clones [`souvik0306/Air-IO`](https://github.com/souvik0306/Air-IO) (`adaptive_loss_function`) and this overlay repo, copies the overlay on top, applies 4 small patches, then runs EuRoC pre-train and 3 fine-tune splits. Runtime: **GPU**.

**Resume.** `experiments/` is a symlink into Drive, so `newest.ckpt` is written there every epoch. Finished phases leave a marker in `DRIVE_DIR/done/`. After a disconnect or quota end, reconnect and run all cells again. New Google account: copy the whole `DRIVE_DIR` folder to that account's Drive at the same path.

**Private repo?** Add a Colab secret `GITHUB_TOKEN` (a fine-grained token with read access to the overlay repo).

In [ ]:
# ===== CONFIG =====
import os
DRIVE_DIR    = "/content/drive/MyDrive/AIIO_colab"
REPO         = "/content/Air-IO"                  # base code (Souvik)
OVERLAY      = "/content/AI-IO-TLab"              # this overlay repo
BASE_URL     = "https://github.com/souvik0306/Air-IO.git"
BASE_BRANCH  = "adaptive_loss_function"
OVERLAY_URL  = "https://github.com/Mndarish/AI-IO-TLab.git"
OVERLAY_BRANCH = "main"

EPOCHS_PRE   = 100        # EuRoC pre-training. Keep identical across sessions.
EPOCHS_FT    = 30         # fine-tuning per split
EXPS         = ["exp0_july_only", "exp1_sepoct_20pct", "exp2_sepoct_40pct"]
FT_FROM_CKPT = "experiments/aiio_euroc/aiio_tf/ckpt/0024.ckpt"   # fine-tune from this EuRoC ckpt (skips pretrain). None = run pretrain.
WANDB_MODE   = "online"   # online | offline | off
WANDB_CONTINUE_RUNS = True   # same W&B run id across sessions. Set False when you switch Google/W&B accounts.

os.environ.update(AIIO_EPOCHS_PRE=str(EPOCHS_PRE), AIIO_EPOCHS_FT=str(EPOCHS_FT),
                  AIIO_WANDB=WANDB_MODE, AIRIO_REPO_DIR=REPO, PBS_O_WORKDIR=REPO)

In [ ]:
# ===== Drive + GPU + helper =====
import subprocess, pathlib, shutil, glob
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE_DIR}/done", exist_ok=True)

def sh(cmd, cwd=None, env=None, check=True):
    """Run a shell command and stream its output."""
    print("\n$", cmd)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO if os.path.isdir(REPO) else cwd, env=env or os.environ,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, executable="/bin/bash")
    for line in p.stdout: print(line, end="")
    rc = p.wait()
    if check and rc != 0: raise RuntimeError(f"failed (exit {rc}): {cmd}")
    return rc

sh("nvidia-smi --query-gpu=name,memory.total --format=csv", cwd="/content")

In [ ]:
# ===== Clone base repo + overlay, copy overlay on top, patch =====
tok = None
try:
    from google.colab import userdata; tok = userdata.get("GITHUB_TOKEN")
except Exception: pass
ov_url = OVERLAY_URL.replace("https://", f"https://{tok}@") if tok else OVERLAY_URL

if not os.path.isdir(REPO):
    sh(f"git clone -q --branch {BASE_BRANCH} {BASE_URL} {REPO}", cwd="/content")
if not os.path.isdir(OVERLAY):
    sh(f"git clone -q --branch {OVERLAY_BRANCH} {ov_url} {OVERLAY}", cwd="/content")
else:
    sh("git pull -q", cwd=OVERLAY)

sh("git log --oneline -1", cwd=REPO); sh("git log --oneline -1", cwd=OVERLAY)

# copy everything except repo metadata / docs / colab helpers
skip = shutil.ignore_patterns(".git", "colab", "docs", "README.md", ".gitignore")
shutil.copytree(OVERLAY, REPO, ignore=skip, dirs_exist_ok=True)
sh("chmod +x *.sh tools/*.sh")
sh(f"python3 {OVERLAY}/colab/apply_patches.py {REPO}")

In [ ]:
# ===== Shims (no apptainer / module on Colab) + python deps =====
sh(f"chmod +x {OVERLAY}/colab/shims/*", cwd="/content")
os.environ["PATH"] = f"{OVERLAY}/colab/shims:" + os.environ["PATH"]

# Colab already has torch/numpy; the pinned versions would break CUDA. Install the rest.
keep = [l for l in open(f"{REPO}/requirements.txt").read().split()
        if not l.startswith(("torch==", "numpy==", "scipy==", "matplotlib==", "opencv"))]
open("/content/requirements-colab.txt", "w").write("\n".join(keep) + "\n")
sh("pip install -q -r /content/requirements-colab.txt")

In [ ]:
# ===== Persist outputs on Drive (symlinks) =====
for d in ["experiments", "result", "results", "results_csv", "orientations", "wandb"]:
    os.makedirs(f"{DRIVE_DIR}/{d}", exist_ok=True)
    p = f"{REPO}/{d}"
    if os.path.islink(p): continue
    if os.path.exists(p): shutil.rmtree(p)
    os.symlink(f"{DRIVE_DIR}/{d}", p)
print(sorted(os.listdir(f"{DRIVE_DIR}/experiments")))

In [ ]:
# ===== Weights & Biases =====
if WANDB_MODE == "online":
    key = None
    try:
        from google.colab import userdata; key = userdata.get("WANDB_API_KEY")
    except Exception: pass
    if not key:
        import getpass; key = getpass.getpass("W&B API key: ")
    os.environ["WANDB_API_KEY"] = key
    os.environ["AIIO_WANDB_CHECK"] = "0"

In [ ]:
# ===== Setup: lab data + EuRoC download, AI-IO configs, self-test =====
sh("bash setup_aiio.sh 2>&1 | tail -40")

hits = glob.glob(f"{REPO}/data/euroc/**/MH_01_easy", recursive=True)
assert hits, "EuRoC not downloaded"
link = f"{REPO}/EuRoC-Dataset"
if os.path.lexists(link): os.remove(link)
os.symlink(os.path.dirname(hits[0]), link)
seqs = "MH_01_easy MH_02_easy MH_03_medium MH_04_difficult MH_05_difficult V1_01_easy V1_02_medium V1_03_difficult V2_01_easy V2_02_medium V2_03_difficult".split()
print("missing EuRoC seqs:", [q for q in seqs if not os.path.isfile(f"{link}/{q}/mav0/imu0/data.csv")] or "none")

In [ ]:
# ===== Runner: skips finished phases, resumes the rest =====
def run_phase(name, script, extra_env=None):
    marker = pathlib.Path(f"{DRIVE_DIR}/done/{name}")
    if marker.exists():
        print(f"[skip] {name} already finished"); return
    env = dict(os.environ, **(extra_env or {}))
    if WANDB_CONTINUE_RUNS:
        env.update(WANDB_RUN_ID=f"aiio_{name}", WANDB_RESUME="allow")
    print(f"[run] {name}")
    sh(f"bash {script}", cwd=REPO, env=env)
    marker.touch(); print(f"[done] {name}")

In [ ]:
# ===== 1. EuRoC pre-training, or reuse an existing EuRoC checkpoint =====
if FT_FROM_CKPT:
    ck = f"{REPO}/{FT_FROM_CKPT}"
    assert os.path.isfile(ck), f"missing {ck}"
    os.environ["AIIO_CHECKPOINT"] = ck
    pathlib.Path(f"{DRIVE_DIR}/done/pretrain").touch()
    print("fine-tuning from", ck)
else:
    run_phase("pretrain", "aiio_pretrain.pbs")

In [ ]:
# ===== 2. Fine-tune + inference + metrics + net-only horizons, per split =====
for e in EXPS:
    run_phase(f"exp_{e}", "aiio_exp.pbs", {"AIIO_EXP": e})

In [ ]:
# ===== 3. Horizon summary (9/12/25 s, net-only) and side-by-side table =====
sh("python3 tools/summarize_horizons.py --mode net --by_group")
sh("python3 tools/compare_horizons.py")

## Extras
**Inference from an uploaded fine-tuned checkpoint** (e.g. trained in another Colab account). Put `best_model.ckpt` in Drive, then run the cell. Results land on Drive under `result/`, `results_csv/`, `orientations/`, `results/horizons_aiio/`.

In [ ]:
EXP  = "exp1_sepoct_20pct"
CKPT = f"{DRIVE_DIR}/exp1_best_model.ckpt"      # None to use the ckpt already in experiments/
for k in ("WANDB_RUN_ID", "WANDB_RESUME"): os.environ.pop(k, None)
sh(f"bash tools/run_inference.sh {EXP} {CKPT or ''}")